# Sales Data Cleaning

**Purpose:** prepare sales transactions for reporting. Run every cell from top to bottom and review the summary before using the output file.

**Expected columns:** sale date, product/item, quantity, unit price, and optionally customer or invoice number.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path('data/sales_raw.csv')  # Change this if your file has another name
OUTPUT_FILE = Path('output/sales_cleaned.csv')
OUTPUT_FILE.parent.mkdir(exist_ok=True)

df = pd.read_excel(DATA_FILE) if DATA_FILE.suffix.lower() in {'.xlsx', '.xls'} else pd.read_csv(DATA_FILE)
print(f'Loaded {len(df):,} sales rows')
df.head()

In [ ]:
# Make column names consistent and show the data structure
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('-', '_'))
print('Columns:', list(df.columns))
df.info()

In [ ]:
# Remove duplicate records and standardize text fields
rows_before = len(df)
df = df.drop_duplicates().copy()
for column in df.select_dtypes(include='object').columns:
    df[column] = df[column].astype('string').str.strip()

# Update these names if the columns in your file are different
df['sale_date'] = pd.to_datetime(df['sale_date'], errors='coerce')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['line_total'] = df['quantity'] * df['unit_price']

In [ ]:
# Keep valid sales only: a date, a product, a positive quantity, and a non-negative price
required = ['sale_date', 'product', 'quantity', 'unit_price']
df = df.dropna(subset=required)
df = df[(df['quantity'] > 0) & (df['unit_price'] >= 0)]

audit = pd.DataFrame({
    'measure': ['input rows', 'duplicates removed', 'missing values remaining', 'clean output rows'],
    'value': [rows_before, rows_before - len(df), int(df.isna().sum().sum()), len(df)]
})
display(audit)
df.head()

In [ ]:
df.to_csv(OUTPUT_FILE, index=False)
print(f'Saved cleaned sales data to: {OUTPUT_FILE}')